In [ ]:
# Databricks notebook source
# MAGIC %md
# MAGIC # 3️⃣ Gold Layer: Labeling & Target Formulation
# MAGIC **Arsitektur**: Medallion Pipeline  
# MAGIC **Dataset**: XAUUSD H1 OHLCV  
# MAGIC **Tujuan**: Deteksi crossover, filter level exhaustion (first touch), aturan target (IMMEDIATE, DELAYED, FAILED, PURE BREAKOUT), validasi invarian T1-T14.

In [ ]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window

# ==============================================================================
# KONFIGURASI & PARAMETER
# ==============================================================================
INPUT_TABLE  = "smt7_research.xauusd.xauusd_liquidity_silver"
OUTPUT_TABLE = "smt7_research.xauusd.xauusd_liquidity_gold"
FUNNEL_TABLE = "smt7_research.xauusd.xauusd_liquidity_dq_funnel"

N_WINDOW = 6
EARLY_K = [1, 2]
TOL_HOURS = 8
OUTCOME_CLASSES = ["IMMEDIATE_SWEEP", "DELAYED_SWEEP", "FAILED_SWEEP", "PURE_BREAKOUT"]

spark.conf.set("spark.sql.session.timeZone", "UTC")

### 1. Load Data Silver

In [ ]:
df_silver = spark.table(INPUT_TABLE)

### 2. Helper Fungsi Pelabelan

In [ ]:
def label_liquidity_events(df, level_col, side, level_name, partition_cols, funnel_dict):
    """
    Fungsi terpadu untuk 4 level: deteksi crossover, filter first touch, filter incomplete N_WINDOW, pelabelan.
    side: 'high' (PDH/PWH) -> <= dan >
    side: 'low' (PDL/PWL) -> >= dan <
    """
    # 1. Deteksi Crossover (Ketat)
    if side == "high":
        cross = (F.col("high") > F.col(level_col)) & (F.col("prev_high") <= F.col(level_col))
    else:
        cross = (F.col("low") < F.col(level_col)) & (F.col("prev_low") >= F.col(level_col))
        
    def inside(col):
        # Inklusif
        return F.col(col) <= F.col(level_col) if side == "high" else F.col(col) >= F.col(level_col)

    ev = df.filter(cross)
    funnel_dict[f"{level_name}_1_crossovers_all"] = ev.count()
    
    # 2. Level Exhaustion (First-Touch)
    w_exhaust = Window.partitionBy(*partition_cols).orderBy("timestamp")
    ev = ev.withColumn("touch_rank", F.row_number().over(w_exhaust)).filter(F.col("touch_rank") == 1)
    
    n_exh = ev.count()
    funnel_dict[f"{level_name}_2_after_exhaustion"] = n_exh
    
    # 3. Filter Jendela Lengkap (T2)
    ev = ev.filter(F.col(f"close_lead_{N_WINDOW}").isNotNull())
    n_comp = ev.count()
    funnel_dict[f"{level_name}_3_dropped_T2"] = n_exh - n_comp
    funnel_dict[f"{level_name}_4_after_T2"] = n_comp
    
    # 4. Window Continuity (T3)
    ev = ev.withColumn("window_hours", (F.unix_timestamp("ts_lead_N") - F.unix_timestamp("timestamp")) / 3600) \
           .withColumn("window_is_continuous", F.when(F.col("window_hours") <= TOL_HOURS, 1).otherwise(0))
           
    # 5. Formulasi Target 4-Class (T1)
    returned_early = inside(f"close_lead_{EARLY_K[0]}")
    for k in EARLY_K[1:]:
        returned_early = returned_early | inside(f"close_lead_{k}")
        
    ev = ev.withColumn("returned_early", returned_early) \
           .withColumn("ended_inside", inside(f"close_lead_{N_WINDOW}"))
           
    ev = ev.withColumn("outcome",
        F.when(F.col("returned_early") & F.col("ended_inside"), "IMMEDIATE_SWEEP")
         .when(~F.col("returned_early") & F.col("ended_inside"), "DELAYED_SWEEP")
         .when(F.col("returned_early") & ~F.col("ended_inside"), "FAILED_SWEEP")
         .when(~F.col("returned_early") & ~F.col("ended_inside"), "PURE_BREAKOUT")
         .otherwise(F.lit(None))
    )
    
    # Kolom turunan / One-hot
    ev = ev.withColumn("is_sweep", F.col("ended_inside").cast("int"))
    for c in OUTCOME_CLASSES:
        ev = ev.withColumn(f"is_{c.lower()}", (F.col("outcome") == c).cast("int"))
        
    return ev.withColumn("level_type", F.lit(level_name)) \
             .withColumn("level_price", F.col(level_col))

### 3. Eksekusi 4 Level & Union

In [ ]:
funnel = {}

df_pdh = label_liquidity_events(df_silver, "PDH", "high", "PDH", ["session_date"], funnel)
df_pdl = label_liquidity_events(df_silver, "PDL", "low",  "PDL", ["session_date"], funnel)
df_pwh = label_liquidity_events(df_silver, "PWH", "high", "PWH", ["week_start"],   funnel)
df_pwl = label_liquidity_events(df_silver, "PWL", "low",  "PWL", ["week_start"],   funnel)

df_gold = df_pdh.unionByName(df_pdl).unionByName(df_pwh).unionByName(df_pwl).orderBy("timestamp")

### 4. Validasi Invarian Kualitas Data (T1 dll)

In [ ]:
# T1: One-hot berjumlah tepat 1
one_hots = [f"is_{c.lower()}" for c in OUTCOME_CLASSES]
k_sum = sum(F.col(c) for c in one_hots)
df_gold = df_gold.withColumn("k", k_sum)
assert df_gold.filter((F.col("k") != 1) | F.col("k").isNull()).count() == 0, "FATAL: Target overlapping / tidak eksklusif!"

# Domain validasi
assert df_gold.filter(F.col("outcome").isNull()).count() == 0, "FATAL: Ada klasifikasi Null."

# Invarian First Touch (T2)
assert df_gold.filter(F.col("level_type").isin("PDH","PDL")).groupBy("session_date","level_type").count().filter(F.col("count")>1).count() == 0
assert df_gold.filter(F.col("level_type").isin("PWH","PWL")).groupBy("week_start","level_type").count().filter(F.col("count")>1).count() == 0

print("=== Rekap Hasil Gold ===")
df_gold.groupBy("level_type", "outcome").count().orderBy("level_type", "outcome").show()

### 5. Simpan Hasil

In [ ]:
# Simpan Tabel Utama
df_gold.drop("k").write.format("delta").mode("overwrite").saveAsTable(OUTPUT_TABLE)

# Simpan Tabel Funnel Quality
funnel_data = [(k, int(v)) for k,v in sorted(funnel.items())]
df_funnel = spark.createDataFrame(funnel_data, ["tahap", "jumlah"])
df_funnel.write.format("delta").mode("overwrite").saveAsTable(FUNNEL_TABLE)

print(f"Gold layer saved to {OUTPUT_TABLE}.")
print(f"DQ Funnel saved to {FUNNEL_TABLE}.")
display(df_gold.limit(5))